**Question:** Does the newest TabPFN release (TabPFN-3.5) with a large context (all training rows) match or beat the production GRU (C3), compared with TabPFN v2 (10k context) and the ELO baseline?

**Assumptions:** Same test matches as ELO/C3/notebook 14 (walk-forward folds 1-4, 14,428 matches pooled); TabPFN-3.5 context = all training rows of the fold (6,132 to 24,528 rows), one seed (42); all predictions isotonic-calibrated on the calibration slice. C3ens is the mean of 5 GRU seeds; C3, TabPFN-3.5 and TabPFN v2 are single-seed.

**Method:** Paired bootstrap by match (2,000 resamples, seed 42, same resample indices for every model), pooled over 4 folds; per-fold Brier; calibration (ECE, reliability diagram) and discrimination (AUC); fold-4 detail separating the effect of the model from the effect of the context size.

**Result:** TabPFN-3.5 is significantly better than ELO and than TabPFN v2, but not better than C3 (pooled dBrier about +0.0007, CI includes 0) and worse than the 5-seed C3ens (about +0.0019, CI above 0). It does not go to production.

# 15 -- Exp D (continued): TabPFN-3.5 (research-only) vs C3, TabPFN v2 and ELO

**Licence note.** TabPFN-3.5 (Prior Labs), used under the TabPFN-3.5 Non-Commercial License; research evaluation only, never used in production (production uses Exp C3 GRU). The model was run locally; all outbound network access was blocked and no hosted API was used.

**Setup.** TabPFN-3.5, variant D-B' (B' features incl. `elo_diff`), context = the *most recent* training rows, either all of them (`full`: 6,132 / 12,264 / 18,396 / 24,528 rows for folds 1-4) or, on fold 4 only, the last 10,000 (`10k`); isotonic calibration as in notebook 14. Fold 4 additionally has the production-compatible pair variant D-B'-par. TabPFN v2 numbers (10k context, seed 42) come from notebook 14 (`../exp_d`); the new runs are read from `../exp_d35`.

**Statistics.** Paired bootstrap by match (both perspective rows of a match travel together), 2,000 resamples, seed 42. Absolute Brier CIs of different models overlap heavily because all models share the same test sample; the **paired dBrier CI is the relevant comparison**. No predictions or data are stored in this notebook.

## Step 0 -- Imports and data

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

sys.path.insert(0, '../scripts')
from exp_d_analysis import (load_folds, add_d35, pool_folds, met, brier_ci, paired_dbrier,
                            reliability, boot_idx)
from summarize_metrics import ece_pp

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

WORKDIR = '..'
B = add_d35(load_folds(WORKDIR), WORKDIR)
V2 = 'p_D-Bprime__last10k__s42'          # TabPFN v2, 10k context, seed 42
MODELS = [('C3ens (5 seeds)', 'p_C3ens'), ('C3', 'p_C3'), ('TabPFN-3.5 (full)', 'p_d35_full'),
          ('TabPFN v2 (10k)', V2), ('ELO', 'p_ELO')]
pool = pool_folds(B)
yp = pool.Label.values
print('folds:', sorted(B), '| pooled matches:', pool.SusretId.nunique(), '| rows:', len(pool))

## Step 1 -- Main table: walk-forward pooled (4 folds), paired against C3

In [ ]:
boots = boot_idx(pool.SusretId.values)
pc3 = pool['p_C3'].values
rows = []
for name, col in MODELS:
    p = pool[col].values
    _, lo, hi = brier_ci(p, yp, boots)
    r = {'model': name, 'Brier': np.mean((p - yp) ** 2), 'CI lo': lo, 'CI hi': hi}
    if col != 'p_C3':
        d, dlo, dhi = paired_dbrier(p, pc3, yp, boots)
        r.update({'dBrier vs C3': d, 'dlo': dlo, 'dhi': dhi})
    else:
        r.update({'dBrier vs C3': 0.0, 'dlo': np.nan, 'dhi': np.nan})
    rows.append(r)
MAIN = pd.DataFrame(rows)
print(MAIN.round(5).to_string(index=False))

Negative dBrier = better than C3. Against C3 (single seed) TabPFN-3.5 is statistically indistinguishable (CI includes 0); TabPFN v2 is worse than C3; ELO is much worse. C3ens averages 5 seeds while C3 / TabPFN are single-seed, so C3ens carries a variance-reduction advantage.

In [ ]:
# Paired against C3ens and TabPFN v2 as well
for ref_name, ref in (('C3ens', 'p_C3ens'), ('TabPFN v2 (10k)', V2)):
    print('paired dBrier vs', ref_name)
    for name, col in MODELS:
        if col == ref:
            continue
        d, lo, hi = paired_dbrier(pool[col].values, pool[ref].values, yp, boots)
        print(f'  {name:20s} {d:+.5f} [{lo:+.5f}, {hi:+.5f}]')

## Step 2 -- Brier by fold

In [ ]:
cols = {'ELO': 'p_ELO', 'C3': 'p_C3', 'C3ens': 'p_C3ens', 'TabPFN v2 (10k)': V2, 'TabPFN-3.5 (full)': 'p_d35_full'}
T = pd.DataFrame([{'fold': k, 'n_matches': B[k].SusretId.nunique(),
                   **{n: np.mean((B[k][c] - B[k].Label) ** 2) for n, c in cols.items()}} for k in sorted(B)])
print(T.round(5).to_string(index=False))
print()
print(T.drop(columns=['fold', 'n_matches']).mean().round(5).rename('mean Brier'))
print()
for n in ['TabPFN-3.5 (full)', 'TabPFN v2 (10k)', 'C3ens', 'ELO']:
    print(f'{n:20s} better than C3 in', int((T[n] < T['C3']).sum()), '/ 4 folds')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(T)); w = 0.2
for i, (n, c) in enumerate([('ELO', '#374151'), ('C3', '#ea580c'), ('TabPFN v2 (10k)', '#2563eb'), ('TabPFN-3.5 (full)', '#16a34a')]):
    ax.bar(x + (i - 1.5) * w, T[n], w, label=n, color=c)
ax.set_xticks(x); ax.set_xticklabels([f'fold {k}' for k in T['fold']])
ax.set_ylim(0.13, 0.19); ax.set_ylabel('Brier score (lower is better)')
ax.set_title('Walk-forward Brier by fold'); ax.legend(ncol=2); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## Step 3 -- Calibration and discrimination

In [ ]:
rows = []
for name, col in MODELS:
    p4, y4_ = B[4][col].values, B[4].Label.values
    rows.append({'model': name, 'ECE fold4 (pp)': ece_pp(p4, y4_), 'AUC fold4': roc_auc_score(y4_, p4),
                 'ECE pooled (pp)': ece_pp(pool[col].values, yp), 'AUC pooled': roc_auc_score(yp, pool[col].values)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True, sharey=True)
style = [('C3', 'p_C3', '#ea580c'), ('TabPFN-3.5 (full)', 'p_d35_full', '#16a34a'),
         ('TabPFN v2 (10k)', V2, '#2563eb'), ('ELO', 'p_ELO', '#374151')]
for ax, (title, d) in zip(axs, (('Fold 4', B[4]), ('Pooled, 4 folds', pool))):
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
    for n, c, col in style:
        mp, ob = reliability(d[c].values, d.Label.values)
        ax.plot(mp, ob, 'o-', ms=4, lw=1.3, color=col, label=n)
    ax.set_title(title); ax.set_xlabel('mean predicted probability'); ax.grid(alpha=0.3)
axs[0].set_ylabel('observed win rate'); axs[0].legend()
plt.tight_layout(); plt.show()

ELO is clearly miscalibrated (pooled ECE 6.4 pp). TabPFN-3.5 has the lowest ECE on fold 4 (1.9 pp vs 3.7 for C3) and pooled (0.96 vs 1.41 pp for C3; C3ens 0.87); in the reliability diagrams all three learned models follow the diagonal closely. Discrimination (AUC) is highest for C3ens, then C3, then TabPFN-3.5, then TabPFN v2, so the remaining Brier gap to C3 comes from discrimination, not calibration.

## Step 4 -- Fold 4 detail: model effect vs context effect

In [ ]:
f4 = B[4]; y4 = f4.Label.values
b4 = boot_idx(f4.SusretId.values)
cmp = [('3.5 full vs C3', 'p_d35_full', 'p_C3'), ('3.5 full vs C3ens', 'p_d35_full', 'p_C3ens'),
       ('3.5 full vs v2 10k', 'p_d35_full', V2),
       ('3.5 10k vs v2 10k   (model effect, same context)', 'p_d35_10k', V2),
       ('3.5 full vs 3.5 10k (context effect, same model)', 'p_d35_full', 'p_d35_10k'),
       ('3.5 10k vs C3', 'p_d35_10k', 'p_C3'), ('3.5 par full vs 3.5 full', 'p_d35_par_full', 'p_d35_full')]
print('Brier fold 4:', {n: round(float(np.mean((f4[c] - y4) ** 2)), 5) for n, c in
      [('3.5 full', 'p_d35_full'), ('3.5 10k', 'p_d35_10k'), ('3.5 par full', 'p_d35_par_full'), ('v2 10k', V2), ('C3', 'p_C3')]})
print()
for n, a, b in cmp:
    mu, lo, hi = paired_dbrier(f4[a].values, f4[b].values, y4, b4)
    print(f'{n:52s} dBrier {mu:+.5f} [{lo:+.5f}, {hi:+.5f}]')

In [ ]:
for name, g in (('SM (men)', 0), ('SF (women)', 1)):
    mk = (f4.liga_tip == g).values
    bt = boot_idx(f4.SusretId.values[mk]); yy = y4[mk]
    br = {n: round(float(np.mean((f4[c].values[mk] - yy) ** 2)), 4) for n, c in
          (('ELO', 'p_ELO'), ('C3', 'p_C3'), ('v2 10k', V2), ('3.5 full', 'p_d35_full'))}
    print(name, 'matches', mk.sum() // 2, br)
    for lab, a, b in (('3.5 full vs C3', 'p_d35_full', 'p_C3'), ('3.5 full vs ELO', 'p_d35_full', 'p_ELO'),
                      ('3.5 full vs v2 10k', 'p_d35_full', V2)):
        print(f'   {lab:20s}', tuple(round(float(x), 5) for x in paired_dbrier(f4[a].values[mk], f4[b].values[mk], yy, bt)))

On fold 4 the larger context does not help: `full` is slightly worse than `10k` (dBrier +0.001, CI includes 0), so the gain of TabPFN-3.5 over v2 is a model effect (same 10k context: dBrier -0.003, CI below 0), not a context-size effect. The pair-compatible variant D-B'-par is somewhat worse than D-B' (CI includes 0). In the women's league (SF, 352 matches) the interval versus C3 is wide and includes 0, as in the men's league (SM).

## Conclusion

- **TabPFN-3.5 (full context) is significantly better than ELO**: pooled dBrier about -0.0155 (CI entirely below 0), better in 4/4 folds.
- **It is better than TabPFN v2 (10k)**: pooled dBrier about -0.0013 (CI below 0), better in 3/4 folds; on fold 4 the effect is a model effect, not a context-size effect.
- **It is not better than the production GRU (C3).** Pooled dBrier vs C3 is about +0.0007 (CI includes 0) and it wins only 2/4 folds. Against the 5-seed C3ens it is worse: about +0.0019 (CI above 0), 0/4 folds.
- It is better calibrated than C3 (lowest ECE) but has slightly lower discrimination (AUC).
- Decision: TabPFN-3.5 does **not** go to production (it is also barred by its non-commercial licence); production remains Exp C3 GRU.

**Limitations.**
- C3ens is the mean of 5 seeds, whereas C3, TabPFN-3.5 and TabPFN v2 are a single seed; the C3ens comparison therefore includes an ensembling advantage.
- TabPFN-3.5 `full` context was run once (seed 42); the `10k` and pair variants only on fold 4.
- The test matches are shared across models, so absolute Brier CIs overlap; only paired CIs are used for comparisons.
- All numbers are isotonic-calibrated; raw outputs were not compared.